<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_08_4_bayesian_hyperparameter_opt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke

**Modul 8: Kaggle-Datensätze**

- Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
– Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).


# Modul 8 Material

* Teil 8.1: Einführung in Kaggle [[Video]](https://www.youtube.com/watch?v=7Mk46fb0Ayg&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=7Mk46fb0Ayg&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.2: Erstellen von Ensembles mit Scikit-Learn und PyTorch [[Video]](https://www.youtube.com/watch?v=przbLRCRL24&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=przbLRCRL24&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.3: Wie sollten Sie Ihr PyTorch-Neuralnetzwerk strukturieren: Hyperparameter [[Video]](https://www.youtube.com/watch?v=YTL2BR4U2Ng&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=YTL2BR4U2Ng&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* **Teil 8.4: Bayesianische Hyperparameteroptimierung für PyTorch** [[Video]](https://www.youtube.com/watch?v=1f4psgAcefU&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1f4psgAcefU&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.5: Kaggle des aktuellen Semesters [[Video]] [[Notebook]](t81_558_class_08_5_kaggle_project.ipynb)

# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab die richtige Version von TensorFlow ausführt.


In [1]:
# Start-up Google CoLab
try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Schön formatierte Zeitzeichenfolge


def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

Note: using Google CoLab
Using device: mps


# Teil 8.4: Bayesianische Hyperparameteroptimierung für PyTorch

Bayesianische Hyperparameteroptimierung ist eine Methode, um Hyperparameter effizienter zu finden als eine Rastersuche. Da jeder Kandidatensatz von Hyperparametern ein erneutes Training des neuronalen Netzwerks erfordert, ist es am besten, die Anzahl der Kandidatensätze auf ein Minimum zu beschränken. Bayesianische Hyperparameteroptimierung erreicht dies, indem ein Modell trainiert wird, um gute Kandidatensätze von Hyperparametern vorherzusagen. [[Cite:snoek2012practical]](https://arxiv.org/pdf/1206.2944.pdf)

- [bayesian-optimization](https://github.com/fmfn/BayesianOptimization)
- [hyperopt](https://github.com/hyperopt/hyperopt)
- [spearmint](https://github.com/JasperSnoek/spearmint)


In [2]:
from scipy.stats import zscore
import pandas as pd
import logging
import os

logging.disable(logging.WARNING)


# Lesen des Datensatzes
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/jh-simple-dataset.csv",
    na_values=["NA", "?"],
)

# Dummies für den Job generieren
df = concat([df, get_dummies(df["job"], prefix="job", dtype=int)], axis=1)
df.drop("job", axis=1, inplace=True)

# Dummies für den Bereich generieren
df = concat([df, get_dummies(df["area"], prefix="area", dtype=int)], axis=1)
df.drop("area", axis=1, inplace=True)

# Fehlende Werte für Einkommen
med = df["income"].median()
df["income"] = df["income"].fillna(med)

# Bereiche standardisieren
df["income"] = zscore(df["income"])
df["aspect"] = zscore(df["aspect"])
df["save_rate"] = zscore(df["save_rate"])
df["age"] = zscore(df["age"])
df["subscriptions"] = zscore(df["subscriptions"])

# In Numpy konvertieren - Klassifizierung
x_columns = df.columns.drop("product").drop("id")
x = df[x_columns].values
dummies = get_dummies(df["product"])  # Einstufung
products = dummies.columns
y = dummies.values

Nachdem wir die Daten nun vorverarbeitet haben, können wir mit der Hyperparameteroptimierung beginnen. Wir beginnen mit der Erstellung einer Funktion, die das Modell auf der Grundlage von nur drei Parametern generiert. Die Bayes-Optimierung arbeitet mit einem Zahlenvektor und nicht mit einem problematischen Begriff wie der Anzahl der Schichten und Neuronen auf jeder Schicht. Um diese komplexe Neuronenstruktur als Vektor darzustellen, verwenden wir mehrere Zahlen, um diese Struktur zu beschreiben.

- **Dropout** – Der Dropout-Prozentsatz für jede Ebene.
- **neuronPct** – Wie viel Prozent unserer festgelegten maximalen Anzahl von 5.000 Neuronen möchten wir verwenden? Dieser Parameter gibt die Gesamtzahl der Neuronen im gesamten Netzwerk an.
- **neuronShrink** - Neuronale Netzwerke beginnen normalerweise mit mehr Neuronen auf der ersten verborgenen Schicht und verringern diese Anzahl dann für weitere Schichten. Dieser Prozentsatz gibt an, um wie viel nachfolgende Schichten basierend auf der vorherigen Schicht verkleinert werden sollen. Wir hören auf, weitere Schichten hinzuzufügen, sobald wir keine Neuronen mehr haben (die Anzahl wird durch neuronPct angegeben).

Diese drei Zahlen definieren die Struktur des neuronalen Netzwerks. Die Befehle im folgenden Code zeigen genau, wie das Programm das Netzwerk aufbaut.


In [3]:
import pandas as pd
import numpy as np
import time
import statistics
from scipy.stats import zscore
from sklearn import metrics
from sklearn.model_selection import StratifiedShuffleSplit
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Konvertieren Sie Daten in PyTorch-Tensoren
x_tensor = torch.FloatTensor(x).to(device)
y_tensor = torch.LongTensor(np.argmax(y, axis=1)).to(
    device
)  # One-Hot in Index konvertieren


class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, dropout, neuronPct, neuronShrink):
        super(NeuralNetwork, self).__init__()

        layers = []
        neuronCount = int(neuronPct * 5000)
        layer = 0

        prev_count = input_dim
        while neuronCount > 25 and layer < 10:
            layers.append(nn.Linear(prev_count, neuronCount))
            prev_count = neuronCount
            layers.append(nn.PReLU())
            layers.append(nn.Dropout(dropout))
            neuronCount = int(neuronCount * neuronShrink)
            layer += 1

        layers.append(nn.Linear(prev_count, y.shape[1]))
        layers.append(nn.Softmax(dim=1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

Wir können diesen Code testen, um zu sehen, wie er basierend auf drei solchen Parametern ein neuronales Netzwerk erstellt.


In [4]:
# Erstellen und Drucken des Modells
model = NeuralNetwork(x.shape[1], 0.2, 0.1, 0.25).to(device)
print(model)

NeuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=47, out_features=500, bias=True)
    (1): PReLU(num_parameters=1)
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=500, out_features=125, bias=True)
    (4): PReLU(num_parameters=1)
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=125, out_features=31, bias=True)
    (7): PReLU(num_parameters=1)
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=31, out_features=7, bias=True)
    (10): Softmax(dim=1)
  )
)


Wir werden nun eine Funktion erstellen, um das neuronale Netzwerk anhand von drei solchen Parametern zu bewerten. Wir verwenden Bootstrapping, da ein Trainingslauf möglicherweise „Pech“ mit den zugewiesenen Zufallsgewichten hat. Wir verwenden diese Funktion, um das neuronale Netzwerk zu trainieren und anschließend zu bewerten.


In [5]:
# Auswertungsfunktion
SPLITS = 2
EPOCHS = 500
PATIENCE = 10


def evaluate_network(learning_rate=1e-3, dropout=0.2, neuronPct=0.1, neuronShrink=0.25):

    boot = StratifiedShuffleSplit(n_splits=SPLITS, test_size=0.1)
    mean_benchmark = []
    epochs_needed = []

    for train, test in boot.split(x, np.argmax(y, axis=1)):
        x_train = x_tensor[train]
        y_train = y_tensor[train]
        x_test = x_tensor[test]
        y_test = y_tensor[test]

        model = NeuralNetwork(
            x.shape[1], dropout=dropout, neuronPct=neuronPct, neuronShrink=neuronShrink
        ).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        dataset_train = TensorDataset(x_train, y_train)
        loader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)

        best_loss = float("inf")
        patience_counter = 0

        for epoch in range(EPOCHS):
            model.train()
            for batch_x, batch_y in loader_train:
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.no_grad():
                outputs_test = model(x_test)
                val_loss = criterion(outputs_test, y_test).item()

            if val_loss < best_loss:
                best_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                epochs_needed.append(epoch)
                break

        # Auswerten
        with torch.no_grad():
            model.eval()
            # Vorhersagen zur Auswertung an die CPU verschieben
            pred = model(x_test).cpu().numpy()
            y_compare = y_test.cpu().numpy()
            score = metrics.log_loss(y_compare, pred)
            mean_benchmark.append(score)

    return -statistics.mean(mean_benchmark)


print(
    evaluate_network(learning_rate=1e-3, dropout=0.2, neuronPct=0.1, neuronShrink=0.25)
)

-4.381666020591725


Sie können jede beliebige Kombination unserer drei Hyperparameter plus der Lernrate ausprobieren, um zu sehen, wie effektiv diese vier Zahlen sind. Natürlich ist es nicht unser Ziel, verschiedene Kombinationen dieser vier Hyperparameter manuell auszuwählen; wir streben eine Automatisierung an.


In [6]:
print(evaluate_network(dropout=0.2, neuronPct=0.1, neuronShrink=0.25))

-5.754028869647901


Wenn wir uns in Colab befinden, müssen wir zuerst das Paket zur Bayesschen Optimierung installieren.


In [7]:
# Ausgabe ausblenden
!pip install bayesian-optimization

Wir werden diesen Prozess nun automatisieren. Wir definieren die Grenzen für jeden dieser vier Hyperparameter und beginnen mit der Bayes-Optimierung. Sobald das Programm abgeschlossen ist, wird die beste gefundene Kombination von Hyperparametern angezeigt. Die **optimize**-Funktion akzeptiert zwei Parameter, die die Dauer des Prozesses erheblich beeinflussen:

- **n_iter** – Wie viele Schritte der Bayes-Optimierung Sie durchführen möchten. Je mehr Schritte, desto wahrscheinlicher ist es, dass Sie ein sinnvolles Maximum finden.
- **init_points**: Wie viele Schritte der zufälligen Erkundung Sie durchführen möchten. Die zufällige Erkundung kann hilfreich sein, indem sie den Erkundungsraum diversifiziert.


In [8]:
from bayes_opt import BayesianOptimization
import time

# Unterdrücken von NaN-Warnungen
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Begrenzter Bereich des Parameterraums
pbounds = {
    "dropout": (0.0, 0.499),
    "learning_rate": (0.0, 0.1),
    "neuronPct": (0.01, 1),
    "neuronShrink": (0.01, 1),
}

optimizer = BayesianOptimization(
    f=evaluate_network,
    pbounds=pbounds,
    verbose=2,  # verbose = 1 druckt nur, wenn ein Maximum
    # wird beachtet, verbose = 0 bedeutet stumm
    random_state=1,
)

start_time = time.time()
optimizer.maximize(
    init_points=10,
    n_iter=20,
)
time_took = time.time() - start_time

print(f"Gesamtlaufzeit: {hms_string(time_took)}")
print(optimizer.max)

|   iter    |  target   |  dropout  | learni... | neuronPct | neuron... |
-------------------------------------------------------------------------
| 1         | -9.167    | 0.2081    | 0.07203   | 0.01011   | 0.3093    |
| 2         | -8.29     | 0.07323   | 0.009234  | 0.1944    | 0.3521    |
| 3         | -9.167    | 0.198     | 0.05388   | 0.425     | 0.6884    |
| 4         | -9.167    | 0.102     | 0.08781   | 0.03711   | 0.6738    |
| 5         | -10.04    | 0.2082    | 0.05587   | 0.149     | 0.2061    |
| 6         | -12.71    | 0.3996    | 0.09683   | 0.3203    | 0.6954    |
| 7         | -8.29     | 0.4373    | 0.08946   | 0.09419   | 0.04866   |
| 8         | -11.84    | 0.08475   | 0.08781   | 0.1074    | 0.4269    |
| 9         | -8.29     | 0.478     | 0.05332   | 0.695     | 0.3224    |
| 10        | -10.04    | 0.3426    | 0.08346   | 0.02811   | 0.7526    |
| 11        | -10.04    | 0.003852  | 0.01619   | 0.7123    | 0.7671    |
| 12        | -12.08    | 0.2208    | 

Wie Sie sehen, hat der Algorithmus insgesamt 30 Iterationen durchgeführt. Diese Gesamtzahl der Iterationen umfasst zehn zufällige und 20 Optimierungsiterationen.
